# INITIAL IMPORT

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

In [2]:
from src.config import Configuration
from src.tetris import TetrisConfiguration

T_CONFIG = TetrisConfiguration(
)

CONFIG = Configuration(
)

pygame-ce 2.5.7 (SDL 2.32.10, Python 3.13.13)


# Data

### Rotation 'r'
- N: north, AKA "0"
- E: east, AKA "R"
- S: south, AKA "2"
- W: west, AKA "L"

### Playfield
- N: Empty
- G: Garbage

### Won and ranking
In case we want to use only won games. Max ranking25000

In [ ]:
raw_df = pd.read_csv(CONFIG.raw_dataset_path)
print(raw_df.columns)

raw_df['subframe'] = raw_df.groupby('game_id').cumcount()
raw_df['playfield'] = raw_df['playfield'].fillna('N')
raw_df['playfield_next'] = raw_df['playfield'].shift(-1).fillna('N')
# raw_df['playfield'].replace('NaN', 'N', inplace=True)
mask_keep_all_but_last = raw_df.duplicated(subset=['game_id'], keep='last')
raw_df = raw_df[mask_keep_all_but_last].copy()


# if no hold and now do -> current = hold and hold = None
# if hold and now is the same -> current = current and hold = hold
# if hold and now is different -> current = hold and hold = current
# 1. Get the previous 'hold' value for each game (defaults to 'N' for the first row)
prev_hold = raw_df.groupby('game_id')['hold'].shift(1).fillna('N')
# 2. Check if the hold value changed compared to the previous frame
hold_changed = prev_hold != raw_df['hold']
# 3. Apply your logic cleanly using np.where(condition, value_if_true, value_if_false)
raw_df['real_current'] = np.where(hold_changed, raw_df['hold'], raw_df['placed'])
raw_df['real_hold'] = np.where(hold_changed, prev_hold, raw_df['hold'])




# NEED TO TRACK COMBOS DONE BY THE PLAYER TO KNOW THE CURRENT B2B AND THE CURRENT COMBO COUNT






# raw_df = raw_df.groupby('game_id')
raw_df = raw_df.sort_values(by=['game_id', 'subframe'])
columns = ['game_id', 'subframe', 'playfield', 'playfield_next', 'real_current', 'real_hold', 'next', 'won', 'rating']
raw_df[columns].to_csv(CONFIG.processed_dataset_path, index=False)

Index(['game_id', 'subframe', 'won', 'playfield', 'x', 'y', 'r', 'placed',
       'hold', 'next', 'cleared', 'garbage_cleared', 'attack', 't_spin', 'btb',
       'combo', 'immediate_garbage', 'incoming_garbage', 'rating', 'glicko',
       'glicko_rd'],
      dtype='str')


### Split

In [4]:
processed_df = pd.read_csv(CONFIG.processed_dataset_path)

# Shuffle by game_id to prevent data leakage (keeping frames of the same game together)
unique_games = pd.Series(processed_df['game_id'].unique()).sample(frac=1, random_state=CONFIG.seed)

# Calculate split indices assuming CONFIG sizes are floats (e.g., 0.1 for 10%)
train_end = int(len(unique_games) * (1 - CONFIG.test_size - CONFIG.val_size))
val_end = train_end + int(len(unique_games) * CONFIG.val_size)

# Partition the shuffled game IDs
train_ids = unique_games.iloc[:train_end]
val_ids = unique_games.iloc[train_end:val_end]
test_ids = unique_games.iloc[val_end:]

# Create the final partitions
train_df = processed_df[processed_df['game_id'].isin(train_ids)].copy()
train_df = train_df.sample(frac=1, random_state=CONFIG.seed).reset_index(drop=True)
val_df = processed_df[processed_df['game_id'].isin(val_ids)].copy()
test_df = processed_df[processed_df['game_id'].isin(test_ids)].copy()

print(f"Train games: {len(train_ids):_}, Val games: {len(val_ids):_}, Test games: {len(test_ids):_}")

train_df.to_csv(CONFIG.tetrio_train, index=False)
val_df.to_csv(CONFIG.tetrio_val, index=False)
test_df.to_csv(CONFIG.tetrio_test, index=False)

Train games: 53_683, Val games: 7_669, Test games: 15_338


# Show the data

In [10]:
game_recod = raw_df.sort_values(by=['game_id', 'subframe'])

frame = 0

In [15]:
from src.tetris import Tetris

playfield = game_recod.iloc[frame]['playfield']
playfield_next = game_recod.iloc[frame]['playfield_next']
next_pieces = game_recod.iloc[frame]['next']
current_piece = game_recod.iloc[frame]['real_current']
hold_piece = game_recod.iloc[frame]['real_hold']
# print(playfield)

game = Tetris(color_map=True, playfield=playfield, next_pieces=next_pieces, active_piece=current_piece, hold_piece=hold_piece)
game.print_state(include_vanish_zone=True)
frame += 1

..........
...J......
...JJJ....
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
...IIII...
Active Piece: J at (3, 20) with rotation SPAWN
Next Piece: ['S', 'O', 'T', 'L', 'S', 'L', 'I', 'O', 'J', 'Z', 'T', 'J', 'T', 'Z']
Hold Piece: N
Can Hold: True


In [20]:
0.01 * 2**10

10.24

In [16]:
from src.tetris import MoveSearcher, Board

def dense_batch_to_bitrows(boards_batch: np.ndarray, width: int) -> np.ndarray:
    """
    Convert dense float32 batch (N, H, W) → bitrows (N, H) uint32.
    boards_batch: shape (N, H, W), values 0.0/1.0
    """
    powers = (1 << np.arange(width, dtype=np.uint32))          # (W,)
    binary = (boards_batch > 0.5).astype(np.uint32)             # (N, H, W)
    return binary.dot(powers)                                    # (N, H) uint32

def find_board_index(board: Board, boards_batch: np.ndarray) -> int:
    """
    Return the index in boards_batch (N, H, W) that matches board.
    Returns -1 if not found.
    
    boards_batch: shape (N, visible_height, width) float32
    """
    N, H, W = boards_batch.shape
    
    # Convert the query board to bitrows
    query = board.b_rows[:H].astype(np.uint32)                  # (H,)
    
    # Convert entire batch to bitrows
    batch_bits = dense_batch_to_bitrows(boards_batch, W)         # (N, H)
    
    # XOR each row against query, OR-reduce across rows — zero means exact match
    diff = (batch_bits ^ query[None, :]).any(axis=1)             # (N,) bool
    
    indices = np.where(~diff)[0]
    return int(indices[0]) if len(indices) > 0 else -1

In [21]:
for i in range(1000):
    searcher = MoveSearcher(game, CONFIG, T_CONFIG)
    _, features = searcher.get_all_features()

import time 
start_time = time.time()
m = 10_000
for i in range(m):
    searcher = MoveSearcher(game, CONFIG, T_CONFIG)
    _, features = searcher.get_all_features()
print(f"Time for {m} iterations: {time.time() - start_time:.2f} seconds, {(time.time() - start_time)/m:.4f} seconds per iteration")

Time for 10000 iterations: 12.08 seconds, 0.0012 seconds per iteration


In [17]:

searcher = MoveSearcher(game, CONFIG, T_CONFIG)
_, features = searcher.get_all_features()
boards_batch = features['boards']   # (128, 24, 10)

board = Board(game.width, game.height, game.vanish_zone, game.color_map, playfield_next)
board.print_board(include_vanish_zone=True)
idx = find_board_index(board, boards_batch)
# idx is the matching index in features, or -1

idx

..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
...ZZ.....
....ZZ....
...IIII...


-1

In [18]:
game = Tetris(color_map=True, playfield=playfield_next, next_pieces=next_pieces, active_piece=current_piece, hold_piece=hold_piece)
game.print_state()

..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
...ZZ.....
....ZZ....
...IIII...
Active Piece: J at (3, 20) with rotation SPAWN
Next Piece: ['S', 'O', 'T', 'L', 'S', 'L', 'I', 'O', 'J', 'Z', 'T', 'J', 'T', 'Z']
Hold Piece: N
Can Hold: True
